# Semifinal 04 - Clean non-retweet tweets and embed them with `intfloat/multilingual-e5-base` (lexical mutants)

In [ ]:
DATA_DIR = "../../data/semifinal"

In [1]:
%pip install datasets tqdm -q

In [3]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

# import tensorflow as tf
# from catboost import CatBoostClassifier, Pool
import torch
import torch.nn.functional as F

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    confusion_matrix
)

import datasets
from transformers import (
    AutoFeatureExtractor,
    AutoModel,
    AutoImageProcessor,
    AutoProcessor,
    AutoTokenizer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


## Helper

In [5]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    # text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # # remove [RE ...]
    # text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # # fix unspaced words
    # new_text = []
    # for word in text.split():
    #     # weird word
    #     if word in ["BEiqnH5xQr", "JN18A"]:
    #         continue
    #     # fix shorten names
    #     elif word == "JK":
    #         new_text.append("jusuf kalla")
    #     elif word.lower() == "jkw":
    #         new_text.append("jokowi")
    #     elif word.lower() in ["bowo", "ayahbowo"]:
    #         new_text.append("prabowo")
    #     elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
    #         new_text.append("pabrowo gibran")
    #     elif word == "MasGBRAN":
    #         new_text.append("gibran")
    #     elif word == "SBY":
    #         new_text.append("susilo bambang yudhoyono")
    #     elif word == "AHY":
    #         new_text.append("agnes haryani")
    #     # keep as is
    #     elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
    #         new_text.append(word)
    #     else:
    #         # fix unspaced words
    #         new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    # text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

In [6]:
def average_pool(last_hidden_states: torch.Tensor,
                 attention_mask: torch.Tensor) -> torch.Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def extract_text_embeddings(
        model,
        feature_extractor,
        text_name="description",
):
    device = model.device
    def pp(batch):
        inputs = feature_extractor(
            text=batch[text_name],
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        # embeddings = model(**inputs).last_hidden_state[:, 0, :].cpu()
        outputs = model(**inputs)
        embeddings = average_pool(outputs.last_hidden_state, inputs.attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return {"embedding": embeddings}
    return pp

def huggingface_text_embedding(
        df,
        text_name="description",
        modelname="intfloat/multilingual-e5-base",
        batched=True,
        batch_size=24,
):
    # initialize huggingface model
    feature_extractor = AutoTokenizer.from_pretrained(modelname)
    model = AutoModel.from_pretrained(modelname, output_hidden_states=True)
    # create huggingface dataset from df
    dataset = datasets.Dataset.from_pandas(df)
    # compute embedding
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extract_fn = extract_text_embeddings(model.to(device), feature_extractor, text_name)
    updated_dataset = dataset.map(extract_fn, batched=batched, batch_size=batch_size)
    df_temp = updated_dataset.to_pandas()
    df_emb = pd.DataFrame()
    df_emb["embedding"] = df_temp["embedding"]
    return df_emb

## Load Data

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/encrypted_merge.csv")
print("Shape of the dataset: ", df.shape)
df.head()

Shape of the dataset:  (9817355, 12)


             created_at                                       username tcode  \
0  2024-01-04T09:57:09Z  @QOS7XYPBfXZWFeLSmdLEt8njUMwwr2Fpel3Cqvh2gW4=    rt   
1  2024-01-04T09:57:09Z  @lSDenDKpcZVnv9txjBcg5qaqxYgVAq/3gTvA8yxPuL4=    rt   
2  2024-01-04T09:57:10Z  @Ykjdr3xs5+WfH9zBQMoAx5fdTeAwmRRm28PFVw5JeJE=    rt   
3  2024-01-04T09:57:10Z  @KQ/OmqgBG/U/OVkvpqAQYYiAThFGxQBtg3J5Vjp4Glk=    rt   
4  2024-01-04T09:57:11Z  @yqECLRUCZgqx8VzEUl430Wj6mfh2SgDYzKwala0bT5o=    rt   

   num_retweets  type  frn_cnt  flw_cnt  sts_cnt             loc  lst_cnt  \
0          1248  twit    266.0    107.0   9687.0             NaN      0.0   
1           195  twit    564.0    303.0  12461.0  indonesia kaya      2.0   
2           116  twit    376.0    156.0   7488.0       Indonesia      1.0   
3          2264  twit    163.0    203.0   2065.0             NaN      0.0   
4          1157  twit      1.0      1.0    798.0             NaN      0.0   

                                             content lan

In [9]:
df.tcode.value_counts()

tcode
rt         6497711
mention    2121754
reply      1197890
Name: count, dtype: int64

In [10]:
df.loc[df.tcode == "mention"].head()

              created_at                                       username  \
21  2024-01-04T09:57:18Z  @YMN1SQke21evlTScXsu3cVvR/l6EPOAYa7EQLb0d5ss=   
40  2024-01-04T09:57:24Z  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=   
41  2024-01-04T09:57:24Z  @vEWivzH0qi/6n1ZjkUSFtLoHE9dC6b5Ymtg8TsyfvQU=   
46  2024-01-04T09:57:26Z  @YBncCoiPad4OqaG+zO7uX04KZRe4WCrZLv05JSRp3XA=   
63  2024-01-04T09:57:32Z  @f0Iyj3PZ8V8hiYInuizI+axftFMAben9UIyKoXGuHYg=   

      tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
21  mention             0  twit    128.0      0.0    819.0   
40  mention             0  twit     10.0      4.0      6.0   
41  mention             0  twit    504.0    650.0  26107.0   
46  mention             0  twit     35.0      2.0    381.0   
63  mention             0  twit     12.0      1.0    219.0   

                       loc  lst_cnt  \
21  Surakarta, Jawa Tengah      0.0   
40                     NaN      0.0   
41                 Riviere      1.0   
46                  

In [11]:
df.duplicated('content').sum()

5754980

In [12]:
df.loc[df.tcode != "rt"].duplicated('content').sum()

323353

In [13]:
df_no_rt = df.loc[df.tcode != "rt"]
df_no_rt.shape

(3319644, 12)

In [14]:
df_no_rt['content_cleaned'] = df_no_rt['content'].apply(remove_noise)
df_no_rt[['content', 'content_cleaned']].head()

<ipython-input-14-8e5b3d40febd>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_rt['content_cleaned'] = df_no_rt['content'].apply(remove_noise)


                                              content  \
21  Mahfud MD dan Ganjar Pranowo menunjukkan bahwa...   
34  @TaMJoOLodMKkoLGb6D7kQt0Xu9UOcQLMRZLaYsmTms8= ...   
38  @9GXTLEV8XmbSNpLSZCd8HnOrQ/uhyesXNsSF+M0gi4Q= ...   
39  @5Tz753nhSrc3MyKUjKPGAKQ59HMT5TnaXVJrYBVFXro= ...   
40  Masyarakat Dander Bojonegoro Bahagia Untuk mem...   

                                      content_cleaned  
21  Mahfud MD dan Ganjar Pranowo menunjukkan bahwa...  
34      Ga capek urus Prabowo Mending urus Ganjar noh  
38  Dalam pembangunan jalan nasional Anies berenca...  
39  Timses Anies jg ada tofa lemon ga kalah rasisn...  
40  Masyarakat Dander Bojonegoro Bahagia Untuk mem...  

In [15]:
df_no_rt[df_no_rt.content_cleaned.str.contains("gemoy")]

                   created_at                                       username  \
40       2024-01-04T09:57:24Z  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=   
1495     2024-01-04T10:05:43Z  @6xr4RjhpHYq8K94m3sixYL8Zddte0sASbMbZ9kkU/l0=   
1928     2024-01-04T10:08:21Z  @12aE5ieuJI3SgY81tYYC6Px2STYYQNhomIMcKig76RU=   
1936     2024-01-04T10:08:24Z  @FH2vQS0hpyBXS9j9EnNrOtbz0c18f6CcWpOfrp0HZg4=   
3055     2024-01-04T10:14:01Z  @kOOD04TvM5zStLQ7yfNp5qi3jJ+NQw+UI59Yz3riSdo=   
...                       ...                                            ...   
9815256  2024-01-06T11:09:53Z  @T6WAqywRYCTUOn6MvWObaAcvegdJhXXpqZB3Rm1fds8=   
9815757  2024-01-06T11:12:00Z  @PQmEIOvKgL8XEcgIHKw6y6dKWF6Gq8dwxW6BzdMa3Lw=   
9816227  2024-01-06T11:13:55Z  @q+lvEls6jdE+anyjuMe9yml4uzcLF1zGlE2ykCimNfc=   
9816295  2024-01-06T11:14:11Z  @MQLiWYThHIrCnkIZoJCWn3XQ3qlxJOu4Wjh66JjNLxM=   
9816740  2024-01-06T11:15:29Z  @m1X98TjkgEorhZautuUNLrWmmXNzXAjPatenphT7Q8M=   

           tcode  num_retweets  type  f

In [16]:
df_no_rt.loc[df_no_rt.index.isin([39736, 39767])]

                 created_at                                       username  \
39736  2024-01-02T13:45:32Z  @426c3Oo4sfDfgAEGhU9pJXAnBsyNuj7jqmBzetbUHko=   
39767  2024-01-02T13:45:43Z  @vydLFy3tRspzvclAop7NSuJo1i2i9YyYHXb7AXTVjBo=   

         tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt         loc  \
39736  mention             0  twit     34.0     19.0    430.0     Kendari   
39767  mention             0  twit     57.0     19.0    449.0  Pagar Alam   

       lst_cnt                                            content lang  \
39736      0.0  Konten gemoy di medsos punya hubungan erat den...   id   
39767      0.0  Konten gemoy yg isinya open BO ternyata lebih ...   id   

                                         content_cleaned  
39736  Konten gemoy di medsos punya hubungan erat den...  
39767  Konten gemoy yg isinya open BO ternyata lebih ...  

In [30]:
tmp_df = df_no_rt.loc[df_no_rt.index.isin([39736, 39767])].copy()
tmp_df["content_cleaned_with_query"] = "query: " + tmp_df["content_cleaned"]
tmp_df

                 created_at                                       username  \
39736  2024-01-02T13:45:32Z  @426c3Oo4sfDfgAEGhU9pJXAnBsyNuj7jqmBzetbUHko=   
39767  2024-01-02T13:45:43Z  @vydLFy3tRspzvclAop7NSuJo1i2i9YyYHXb7AXTVjBo=   

         tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt         loc  \
39736  mention             0  twit     34.0     19.0    430.0     Kendari   
39767  mention             0  twit     57.0     19.0    449.0  Pagar Alam   

       lst_cnt                                            content lang  \
39736      0.0  Konten gemoy di medsos punya hubungan erat den...   id   
39767      0.0  Konten gemoy yg isinya open BO ternyata lebih ...   id   

                                         content_cleaned  \
39736  Konten gemoy di medsos punya hubungan erat den...   
39767  Konten gemoy yg isinya open BO ternyata lebih ...   

                              content_cleaned_with_query  
39736  query: Konten gemoy di medsos punya hubungan e...  
39767  quer

In [17]:
tmp_df_embed = huggingface_text_embedding(
    tmp_df,
    text_name="content_cleaned",
    modelname="intfloat/multilingual-e5-base"
)

tmp_df_embed

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

                                           embedding
0  [0.33689535, 0.76808804, -0.28155446, 0.333340...
1  [0.15113582, 0.69785434, -0.59094644, -0.09351...

In [24]:
cosine_similarity(tmp_df_embed.embedding.values[0].reshape(1, -1), tmp_df_embed.embedding.values[1].reshape(1, -1))

array([[0.9618945]], dtype=float32)

In [31]:
tmp_df_embed = huggingface_text_embedding(
    tmp_df,
    text_name="content_cleaned_with_query",
    modelname="intfloat/multilingual-e5-base"
)

tmp_df_embed

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

                                           embedding
0  [0.22256294, 0.97570914, -0.06416632, 0.287401...
1  [0.07966852, 1.0313897, -0.35427007, -0.067947...

In [32]:
cosine_similarity(tmp_df_embed.embedding.values[0].reshape(1, -1), tmp_df_embed.embedding.values[1].reshape(1, -1))

array([[0.96626467]], dtype=float32)

In [34]:
# with hf embedding extraction method
tmp_df_embed = huggingface_text_embedding(
    tmp_df,
    text_name="content_cleaned_with_query",
    modelname="intfloat/multilingual-e5-base"
)

tmp_df_embed

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

                                           embedding
0  [0.0099453805, 0.05927703, -0.011440859, 0.017...
1  [-0.0025306814, 0.06408328, -0.027115844, -0.0...

In [37]:
cosine_similarity(tmp_df_embed.embedding.values[0].reshape(1, -1), tmp_df_embed.embedding.values[1].reshape(1, -1))

array([[0.95207214]], dtype=float32)

In [43]:
tmp_embed = tmp_df_embed.embedding.values[0]
tmp_embed2 = tmp_df_embed.embedding.values[1]

In [45]:
tmp_scores = (tmp_embed @ tmp_embed2.T) * 100
tmp_scores

95.20719647407532

In [46]:
df_no_rt['content_cleaned_with_query'] = " query: " + df_no_rt['content_cleaned']
df_no_rt.head()

<ipython-input-46-d9bdf206ed9b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_rt['content_cleaned_with_query'] = " query: " + df_no_rt['content_cleaned']


              created_at                                       username  \
21  2024-01-04T09:57:18Z  @YMN1SQke21evlTScXsu3cVvR/l6EPOAYa7EQLb0d5ss=   
34  2024-01-04T09:57:22Z  @7WZM6I5Q/iYrH1SMgclp34S65jhXMMyhXdvYbR3TO9s=   
38  2024-01-04T09:57:23Z  @5bcvctYrjL1R7+juB8TjhVbDqjgCOCKLMJVerK5CqF4=   
39  2024-01-04T09:57:23Z  @frAtGFoe5QExA7+rfzOX6vqrbItjXQdZkORwO0DWHW0=   
40  2024-01-04T09:57:24Z  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=   

      tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
21  mention             0  twit    128.0      0.0    819.0   
34    reply             0  twit    347.0    207.0   6378.0   
38    reply             0  twit    153.0    124.0    319.0   
39    reply             0  twit    278.0     94.0   8740.0   
40  mention             0  twit     10.0      4.0      6.0   

                       loc  lst_cnt  \
21  Surakarta, Jawa Tengah      0.0   
34                semarang      1.0   
38                     NaN      0.0   
39                  

In [47]:
df_no_rt.reset_index(drop=False, inplace=True)
df_no_rt.head()

   index            created_at                                       username  \
0     21  2024-01-04T09:57:18Z  @YMN1SQke21evlTScXsu3cVvR/l6EPOAYa7EQLb0d5ss=   
1     34  2024-01-04T09:57:22Z  @7WZM6I5Q/iYrH1SMgclp34S65jhXMMyhXdvYbR3TO9s=   
2     38  2024-01-04T09:57:23Z  @5bcvctYrjL1R7+juB8TjhVbDqjgCOCKLMJVerK5CqF4=   
3     39  2024-01-04T09:57:23Z  @frAtGFoe5QExA7+rfzOX6vqrbItjXQdZkORwO0DWHW0=   
4     40  2024-01-04T09:57:24Z  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=   

     tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
0  mention             0  twit    128.0      0.0    819.0   
1    reply             0  twit    347.0    207.0   6378.0   
2    reply             0  twit    153.0    124.0    319.0   
3    reply             0  twit    278.0     94.0   8740.0   
4  mention             0  twit     10.0      4.0      6.0   

                      loc  lst_cnt  \
0  Surakarta, Jawa Tengah      0.0   
1                semarang      1.0   
2                     NaN      0

In [55]:
df_no_rt.to_csv(f"{DATA_DIR}/encrypted_merge_no_rt.csv", index=False)

In [7]:
df_no_rt = pd.read_csv(f"{DATA_DIR}/encrypted_merge_no_rt.csv")
df_no_rt.head()

   index            created_at                                       username  \
0     21  2024-01-04T09:57:18Z  @YMN1SQke21evlTScXsu3cVvR/l6EPOAYa7EQLb0d5ss=   
1     34  2024-01-04T09:57:22Z  @7WZM6I5Q/iYrH1SMgclp34S65jhXMMyhXdvYbR3TO9s=   
2     38  2024-01-04T09:57:23Z  @5bcvctYrjL1R7+juB8TjhVbDqjgCOCKLMJVerK5CqF4=   
3     39  2024-01-04T09:57:23Z  @frAtGFoe5QExA7+rfzOX6vqrbItjXQdZkORwO0DWHW0=   
4     40  2024-01-04T09:57:24Z  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=   

     tcode  num_retweets  type  frn_cnt  flw_cnt  sts_cnt  \
0  mention             0  twit    128.0      0.0    819.0   
1    reply             0  twit    347.0    207.0   6378.0   
2    reply             0  twit    153.0    124.0    319.0   
3    reply             0  twit    278.0     94.0   8740.0   
4  mention             0  twit     10.0      4.0      6.0   

                      loc  lst_cnt  \
0  Surakarta, Jawa Tengah      0.0   
1                semarang      1.0   
2                     NaN      0

In [8]:
df_no_rt.shape

(3319644, 15)

In [9]:
df_no_rt_part1 = df_no_rt.iloc[:1000000]
df_no_rt_part1.shape

(1000000, 15)

In [10]:
# with hf embedding extraction method
embeddings_df = huggingface_text_embedding(
    df_no_rt_part1,
    text_name="content_cleaned_with_query",
    modelname="intfloat/multilingual-e5-base",
    batch_size=16
)

df_no_rt_part1["embedding"] = embeddings_df["embedding"]
with open(f'{DATA_DIR}/embedding_part1.pkl', 'wb') as f:
    pickle.dump(df_no_rt_part1["embedding"].values, f)
df_no_rt_part1.head()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
import gc

del embeddings_df
del df_no_rt_part1
gc.collect()

In [ ]:
df_no_rt_part2 = df_no_rt.iloc[1000000:2000000]
df_no_rt_part2.shape

In [ ]:
# with hf embedding extraction method
embeddings_df = huggingface_text_embedding(
    df_no_rt_part2,
    text_name="content_cleaned_with_query",
    modelname="intfloat/multilingual-e5-base",
    batch_size=16
)

df_no_rt_part2["embedding"] = embeddings_df["embedding"]
with open(f'{DATA_DIR}/embedding_part2.pkl', 'wb') as f:
    pickle.dump(df_no_rt_part2["embedding"].values, f)
df_no_rt_part2.head()

In [ ]:
import gc

del embeddings_df
del df_no_rt_part2
gc.collect()

In [ ]:
df_no_rt_part3 = df_no_rt.iloc[2000000:]
df_no_rt_part3.shape

In [ ]:
# with hf embedding extraction method
embeddings_df = huggingface_text_embedding(
    df_no_rt_part3,
    text_name="content_cleaned_with_query",
    modelname="intfloat/multilingual-e5-base",
    batch_size=16
)

df_no_rt_part3["embedding"] = embeddings_df["embedding"]
with open(f'{DATA_DIR}/embedding_part3.pkl', 'wb') as f:
    pickle.dump(df_no_rt_part3["embedding"].values, f)
df_no_rt_part3.head()